# Dataset
Every speech chunk of `outputs/chunks/<podcast>/` (made in `05_chunks`), with its text, in one folder, resampled to `OUT_SR`.
Skipped: chunks without a speaker (pauses, music, and the little speech the diarization missed) and chunks without text.
Output: `dataset/chunks/` + `dataset/metadata.json`. Each run rebuilds the dataset from scratch.

`metadata.json`: `sample_rate`, `podcasts` (number → title), and per chunk `file, speaker, text, duration, start, end`.
`speaker` has the podcast number in front, because `speaker_0` is only unique inside one podcast. `start` / `end` say where the chunk is in the podcast audio.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path

import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
chunks_root = root / "outputs" / "chunks"
dataset_dir = root / "dataset"
wav_dir = dataset_dir / "chunks"
OUT_SR = 48000  # every chunk is resampled to this (05 keeps each podcast's own rate: 24 kHz, abuser 44.1 kHz)
# sharp filter: torchaudio's default one takes ~3-4 dB off the top of the band, this one keeps it
RESAMPLE = dict(lowpass_filter_width=64, rolloff=0.99, resampling_method="sinc_interp_kaiser", beta=14.769656459379492)
ABSOLUTE_PATHS = False  # False: file = "chunks/<name>.wav" (relative to dataset/), True: full path

podcasts, items, skipped = [], [], Counter()  # items: (podcast number, source wav, chunk)
for podcast_dir in sorted(p for p in chunks_root.iterdir() if p.is_dir()):
    path = podcast_dir / "chunks.json"
    if not path.exists():
        print("skip (no chunks.json, run 05_chunks):", podcast_dir.name)
        continue
    podcasts.append(podcast_dir.name)
    for c in json.loads(path.read_text(encoding="utf-8"))["chunks"]:
        if c["speaker"] is None:
            skipped["no speaker, has text: speech the diarization missed" if c["text"] else
                    "no speaker, no text: pause, music"] += 1
        elif c["text"] is None:
            skipped["no text: data/alignments had no file for this podcast"] += 1
        elif not c["text"]:
            skipped["speaker but no word: mostly < 1 s (breath, laugh, 'mm')"] += 1
        else:
            items.append((len(podcasts), podcast_dir / c["file"], c))


def new_name(num, src):
    """File names repeat across podcasts (every one has 0000_speaker_0.wav) -> podcast number in front."""
    return f"{num:03d}_{src.name}"


missing = [src for _, src, _ in items if not src.exists()]
names = Counter(new_name(num, src) for num, src, _ in items)
assert not missing, f"{len(missing)} chunks in a chunks.json have no wav, e.g. {missing[:3]}"
assert max(names.values(), default=1) == 1, "two chunks would get the same name"
print(f"{len(items)} chunks from {len(podcasts)} podcasts, {sum(c['duration'] for *_, c in items) / 3600:.2f} h")
for why, n in skipped.items():
    print(f"  skipped {n} chunks: {why}")
for num, title in enumerate(podcasts, 1):
    print(f"  {num:03d}  {title}")

13697 chunks from 31 podcasts, 40.39 h
  skipped 1118 chunks: speaker but no word: mostly < 1 s (breath, laugh, 'mm')
  skipped 134 chunks: no speaker, has text: speech the diarization missed
  skipped 112 chunks: no speaker, no text: pause, music
  001  18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak
  002  2026-yilda Google'ga ishga kirish XARITASI
  003  28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili
  004  29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi
  005  31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda
  006  40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son
  007  50 yoshida ham studentlikni tanlaydigan odam bilan suhbat
  008  99% odamlar buni eplay olmaydi - Iqtibos podcast #27
  009  AQLI PASTlar avlodi (bo'lib qolyapmiz)
  010  Afsus qilmaslik uchun bugun nima qilish kerak - Hasanxon Yahyo Abdulmajid
  011  Ch

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wav_dir.mkdir(parents=True, exist_ok=True)
for old in [*wav_dir.glob("*.wav"), dataset_dir / "metadata.json", dataset_dir / "metadata.jsonl"]:
    old.unlink(missing_ok=True)  # an earlier run (metadata.jsonl: the old format)

rows = []
for num, src, c in tqdm(items, unit="file"):
    dst = wav_dir / new_name(num, src)
    x, sr = sf.read(src, dtype="float32", always_2d=True)
    if sr != OUT_SR:  # resample each channel, 16-bit wav like the source
        x = AF.resample(torch.from_numpy(x.T.copy()), sr, OUT_SR, **RESAMPLE).T.numpy().clip(-1.0, 1.0)
    sf.write(dst, x, OUT_SR, subtype="PCM_16")
    rows.append({
        "file": str(dst) if ABSOLUTE_PATHS else f"{wav_dir.name}/{dst.name}",
        "speaker": f"{num:03d}_{c['speaker']}",
        "text": c["text"],
        "duration": round(len(x) / OUT_SR, 3),
        "start": c["start"],
        "end": c["end"],
    })

metadata = {
    "sample_rate": OUT_SR,
    "podcasts": {f"{num:03d}": title for num, title in enumerate(podcasts, 1)},
    "chunks": rows,
}
tmp = dataset_dir / "metadata.json.tmp"
tmp.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(tmp, dataset_dir / "metadata.json")  # written last: no metadata.json = the run did not finish
print("saved", len(rows), "chunks to", wav_dir, "and", dataset_dir / "metadata.json")

100%|██████████| 13697/13697 [02:01<00:00, 113.16file/s]

saved 13697 chunks to /Users/macbook/Documents/IT/projects/tts/dataset/chunks and /Users/macbook/Documents/IT/projects/tts/dataset/metadata.json


In [3]:
# check: metadata and folder match, and a summary of the dataset
meta = json.loads((dataset_dir / "metadata.json").read_text(encoding="utf-8"))
rows = meta["chunks"]
wavs = {p.name for p in wav_dir.glob("*.wav")}
listed = {Path(m["file"]).name for m in rows}
print("rows:", len(rows), "| wavs:", len(wavs), "| wav without row:", len(wavs - listed),
      "| row without wav:", len(listed - wavs), "| row without text:", sum(not m["text"] for m in rows))
total = sum(m["duration"] for m in rows)
rates = Counter(sf.info(str(wav_dir / name)).samplerate for name in listed & wavs)
print(f"total {int(total // 3600)}h {int(total % 3600 // 60):02d}m, {len({m['speaker'] for m in rows})} speakers, "
      f"{sum(len(m['text'].split()) for m in rows)} words, sample rates {dict(rates)}")
for lo, hi in zip([0, 1, 2, 5, 10, 20], [1, 2, 5, 10, 20, 30.5]):
    part = [m["duration"] for m in rows if lo <= m["duration"] < hi]
    print(f"  {lo:>2}-{hi:<4} s: {len(part):5d} chunks, {sum(part) / 3600:5.2f} h")

rows: 13697 | wavs: 13697 | wav without row: 0 | row without wav: 0 | row without text: 0
total 40h 23m, 120 speakers, 320155 words, sample rates {48000: 13697}
   0-1    s:  1114 chunks,  0.24 h
   1-2    s:  2467 chunks,  0.99 h
   2-5    s:  2796 chunks,  2.49 h
   5-10   s:  1952 chunks,  3.92 h
  10-20   s:  2059 chunks,  8.35 h
  20-30.5 s:  3309 chunks, 24.40 h
